# Improved LightGBM — Walmart Store Sales Forecasting

This notebook fixes the mismatch where local validation was approximately **1,500–1,800 WMAE** but the Kaggle public score was approximately **25,000**.

## Root cause

The registered pipeline performed feature engineering that sorted test rows by `Store` and `Date`. It then returned the resulting predictions with the **original input index without restoring the original row order**. Consequently, a prediction computed for one `(Store, Dept, Date)` row could be written to another row in the submission. A tiny `head(20)` sanity check did not detect this because it did not compare row identities before and after preprocessing.

## Other improvements

1. Validation is one exact **39-week direct forecast**, matching Kaggle.
2. Training origins are restricted so their target horizons finish before validation begins.
3. The model predicts a correction over a strong lag-52 seasonal baseline.
4. Holiday rows receive weight 5 during both training and early stopping.
5. A validation-selected shrinkage factor controls how much of LightGBM's correction is used.
6. Prediction corrections are clipped to robust training quantiles to prevent extreme forecasts.
7. Full-test submission generation preserves the exact original test order and verifies IDs.
8. The registered MLflow pipeline stores and restores row positions explicitly.

The notebook reuses the project's `features.py`, `wmae.py`, and `mlflow_setup.py`.

## 1. Imports and configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.pyfunc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid

from features import (
    load_raw_data,
    merge_all,
    build_all_features,
    add_origin_style_features,
    build_origin_training_frame,
    ALL_FEATURES_ORIGIN,
    CATBOOST_CAT_FEATURES,
    TARGET,
)
from wmae import wmae
from mlflow_setup import init_mlflow

DATA_DIR = "."
RANDOM_SEED = 42
HORIZON = 39
ORIGIN_STRIDE = 2

# These are deliberately excluded because they are unreliable for true extrapolation:
# - Year=2013 is unseen during training.
# - CPI/Unemployment are unavailable for part of the Kaggle horizon.
DROP_FEATURES = {"Year", "CPI", "Unemployment"}

# Keep origin rolling statistics: they are genuinely known at forecast time and are
# computed in exactly the same frozen manner for historical origins and Kaggle test.
LGBM_FEATURES = [c for c in ALL_FEATURES_ORIGIN if c not in DROP_FEATURES]

# Additional stable features created below.
EXTRA_FEATURES = [
    "Week_Sin", "Week_Cos",
    "Month_Sin", "Month_Cos",
    "Horizon_Sin", "Horizon_Cos",
    "Seasonal_Baseline",
    "Lag52_Missing",
]
LGBM_FEATURES = LGBM_FEATURES + EXTRA_FEATURES
CAT_FEATURES = [c for c in CATBOOST_CAT_FEATURES if c in LGBM_FEATURES]

np.random.seed(RANDOM_SEED)
print("LightGBM:", lgb.__version__)
print("Feature count:", len(LGBM_FEATURES))
print("Categorical features:", CAT_FEATURES)

LightGBM: 4.6.0
Feature count: 43
Categorical features: ['Store', 'Dept', 'Type_Enc', 'Month', 'WeekOfYear', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'IsHoliday']


## 2. Load and merge competition data

In [2]:
train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
train_merged, test_merged = merge_all(
    train_raw.copy(), test_raw.copy(), features_raw.copy(), stores_raw.copy()
)

train_merged["Date"] = pd.to_datetime(train_merged["Date"])
test_merged["Date"] = pd.to_datetime(test_merged["Date"])

# Preserve competition order from the beginning.
test_merged["_input_row"] = np.arange(len(test_merged), dtype=np.int64)

print("train:", train_merged.shape)
print("test :", test_merged.shape)
print("train dates:", train_merged["Date"].min().date(), "->", train_merged["Date"].max().date())
print("test dates :", test_merged["Date"].min().date(), "->", test_merged["Date"].max().date())
print("negative sales rows retained:", int((train_merged[TARGET] < 0).sum()))

train: (421570, 16)
test : (115064, 16)
train dates: 2010-02-05 -> 2012-10-26
test dates : 2012-11-02 -> 2013-07-26
negative sales rows retained: 1285


## 3. Honest 39-week validation block

The last 39 labeled weeks are treated exactly like the Kaggle test horizon. The validation origin is the last known date immediately before those weeks. No validation target is included in the historical data used to construct its features.

In [3]:
all_train_dates = pd.Index(sorted(train_merged["Date"].unique()))
assert len(all_train_dates) > HORIZON + 52

validation_dates = all_train_dates[-HORIZON:]
validation_start = pd.Timestamp(validation_dates[0])
validation_origin = pd.Timestamp(all_train_dates[-HORIZON - 1])

history_for_validation = train_merged[train_merged["Date"] <= validation_origin].copy()
validation_raw = train_merged[train_merged["Date"].isin(validation_dates)].copy()
validation_raw["_input_row"] = np.arange(len(validation_raw), dtype=np.int64)

validation_base = build_all_features(validation_raw, add_lags=False)
validation_frame = add_origin_style_features(
    validation_base,
    history_for_validation,
    origin_date=validation_origin,
)

# Replace row-count shift(52) with an exact 364-day date lookup. This remains
# correct for sparse Store-Dept series and for arbitrary inference subsets.
_exact_lag = history_for_validation[["Store", "Dept", "Date", TARGET]].copy()
_exact_lag["Date"] = _exact_lag["Date"] + pd.Timedelta(weeks=52)
_exact_lag = _exact_lag.rename(columns={TARGET: "_Exact_Lag52"})
validation_frame = validation_frame.merge(
    _exact_lag, on=["Store", "Dept", "Date"], how="left", validate="many_to_one"
)
validation_frame["Sales_Lag52_Origin"] = validation_frame["_Exact_Lag52"].combine_first(
    validation_frame["Sales_Lag52_Origin"]
)
validation_frame = validation_frame.drop(columns="_Exact_Lag52")

# Add stable periodic encodings.
validation_frame["Week_Sin"] = np.sin(2 * np.pi * validation_frame["WeekOfYear"] / 52.0)
validation_frame["Week_Cos"] = np.cos(2 * np.pi * validation_frame["WeekOfYear"] / 52.0)
validation_frame["Month_Sin"] = np.sin(2 * np.pi * validation_frame["Month"] / 12.0)
validation_frame["Month_Cos"] = np.cos(2 * np.pi * validation_frame["Month"] / 12.0)
validation_frame["Horizon_Sin"] = np.sin(2 * np.pi * validation_frame["Forecast_Horizon"] / HORIZON)
validation_frame["Horizon_Cos"] = np.cos(2 * np.pi * validation_frame["Forecast_Horizon"] / HORIZON)

# Strong seasonal baseline. Rolling means are only fallbacks for sparse series.
validation_frame["Lag52_Missing"] = validation_frame["Sales_Lag52_Origin"].isna().astype(int)
validation_frame["Seasonal_Baseline"] = (
    validation_frame["Sales_Lag52_Origin"]
    .fillna(validation_frame["Sales_Roll_52w_Mean_Origin"])
    .fillna(validation_frame["Sales_Roll_26w_Mean_Origin"])
    .fillna(validation_frame["Sales_Roll_13w_Mean_Origin"])
    .fillna(history_for_validation[TARGET].median())
)

print("validation origin:", validation_origin.date())
print("validation range :", validation_frame["Date"].min().date(), "->", validation_frame["Date"].max().date())
print("validation weeks :", validation_frame["Date"].nunique())
print("validation rows  :", len(validation_frame))
assert validation_frame["Date"].nunique() == HORIZON
assert validation_frame["Date"].min() > validation_origin

validation origin: 2012-01-27
validation range : 2012-02-03 -> 2012-10-26
validation weeks : 39
validation rows  : 115588


## 4. Build training origins without overlap

Only data available up to the validation origin is passed to the origin-frame builder. Therefore every training example's 39-week target horizon ends before the validation block.

In [4]:
origin_train = build_origin_training_frame(
    history_for_validation,
    horizon_weeks=HORIZON,
    origin_stride_weeks=ORIGIN_STRIDE,
    min_history_weeks=52,
    require_full_horizon=True,
)

_exact_lag = history_for_validation[["Store", "Dept", "Date", TARGET]].copy()
_exact_lag["Date"] = _exact_lag["Date"] + pd.Timedelta(weeks=52)
_exact_lag = _exact_lag.rename(columns={TARGET: "_Exact_Lag52"})
origin_train = origin_train.merge(
    _exact_lag, on=["Store", "Dept", "Date"], how="left", validate="many_to_one"
)
origin_train["Sales_Lag52_Origin"] = origin_train["_Exact_Lag52"].combine_first(
    origin_train["Sales_Lag52_Origin"]
)
origin_train = origin_train.drop(columns="_Exact_Lag52")

origin_train["Week_Sin"] = np.sin(2 * np.pi * origin_train["WeekOfYear"] / 52.0)
origin_train["Week_Cos"] = np.cos(2 * np.pi * origin_train["WeekOfYear"] / 52.0)
origin_train["Month_Sin"] = np.sin(2 * np.pi * origin_train["Month"] / 12.0)
origin_train["Month_Cos"] = np.cos(2 * np.pi * origin_train["Month"] / 12.0)
origin_train["Horizon_Sin"] = np.sin(2 * np.pi * origin_train["Forecast_Horizon"] / HORIZON)
origin_train["Horizon_Cos"] = np.cos(2 * np.pi * origin_train["Forecast_Horizon"] / HORIZON)
origin_train["Lag52_Missing"] = origin_train["Sales_Lag52_Origin"].isna().astype(int)
origin_train["Seasonal_Baseline"] = (
    origin_train["Sales_Lag52_Origin"]
    .fillna(origin_train["Sales_Roll_52w_Mean_Origin"])
    .fillna(origin_train["Sales_Roll_26w_Mean_Origin"])
    .fillna(origin_train["Sales_Roll_13w_Mean_Origin"])
    .fillna(history_for_validation[TARGET].median())
)

# Predict a correction over the baseline instead of predicting the full sales level.
origin_train["Residual_Target"] = origin_train[TARGET] - origin_train["Seasonal_Baseline"]
validation_frame["Residual_Target"] = validation_frame[TARGET] - validation_frame["Seasonal_Baseline"]

print("origin training frame:", origin_train.shape)
print("training origins:", origin_train["Forecast_Origin_Date"].nunique())
print("latest train target:", origin_train["Date"].max().date())
print("first validation target:", validation_frame["Date"].min().date())
assert origin_train["Date"].max() <= validation_origin
assert origin_train["Date"].max() < validation_frame["Date"].min()

origin training frame: (805497, 53)
training origins: 7
latest train target: 2012-01-20
first validation target: 2012-02-03


## 5. Feature and row-order diagnostics

In [5]:
missing_train = [c for c in LGBM_FEATURES if c not in origin_train.columns]
missing_val = [c for c in LGBM_FEATURES if c not in validation_frame.columns]
assert not missing_train, f"Missing training features: {missing_train}"
assert not missing_val, f"Missing validation features: {missing_val}"

# This explicitly demonstrates why the previous pipeline was dangerous:
# build_all_features() may reorder rows, so the positional key must be restored.
validation_reordered = not np.array_equal(
    validation_frame["_input_row"].to_numpy(),
    np.arange(len(validation_frame), dtype=np.int64),
)
print("Feature engineering changed validation row order:", validation_reordered)
print("The fixed inference path always sorts back by _input_row before returning predictions.")

comparison_cols = [
    "Sales_Lag52_Origin", "Sales_Roll_52w_Mean_Origin",
    "MarkDown_Total", "Temperature", "Fuel_Price", "Forecast_Horizon"
]
comparison = pd.DataFrame({
    "train_na": origin_train[comparison_cols].isna().mean(),
    "val_na": validation_frame[comparison_cols].isna().mean(),
    "train_mean": origin_train[comparison_cols].mean(),
    "val_mean": validation_frame[comparison_cols].mean(),
})
comparison

Feature engineering changed validation row order: True
The fixed inference path always sorts back by _input_row before returning predictions.


,train_na,val_na,train_mean,val_mean
Sales_Lag52_Origin,0.024076,0.017156,16302.620891,15727.063589
Sales_Roll_52w_Mean_Origin,0.003636,0.001730,16005.399868,15946.324097
MarkDown_Total,0.000000,0.000000,3607.759436,17106.012482
Temperature,0.000000,0.000000,64.294001,65.135183
Fuel_Price,0.000000,0.000000,3.655220,3.745333
Forecast_Horizon,0.000000,0.000000,20.037893,19.978285


## 6. Prepare LightGBM matrices and metric-aligned weights

In [6]:
X_train = origin_train[LGBM_FEATURES].copy()
X_val = validation_frame[LGBM_FEATURES].copy()

for c in CAT_FEATURES:
    train_categories = sorted(pd.concat([X_train[c], X_val[c]], ignore_index=True).dropna().unique())
    X_train[c] = pd.Categorical(X_train[c], categories=train_categories)
    X_val[c] = pd.Categorical(X_val[c], categories=train_categories)

y_train = origin_train["Residual_Target"].astype(float)
y_val_residual = validation_frame["Residual_Target"].astype(float)
y_val_level = validation_frame[TARGET].astype(float)
val_baseline = validation_frame["Seasonal_Baseline"].astype(float).to_numpy()

train_weights = np.where(origin_train["IsHoliday"].astype(bool), 5.0, 1.0)
val_weights = np.where(validation_frame["IsHoliday"].astype(bool), 5.0, 1.0)

baseline_wmae = wmae(
    y_val_level.to_numpy(),
    val_baseline,
    validation_frame["IsHoliday"].to_numpy(),
)
print(f"Lag-52 seasonal baseline WMAE: {baseline_wmae:,.2f}")

Lag-52 seasonal baseline WMAE: 1,814.40


## 7. Hyperparameter search

The search is intentionally compact. Early stopping uses holiday-weighted validation MAE, and candidate selection uses the competition WMAE after converting residual predictions back to sales levels.

In [7]:
param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.02, 0.04],
    "min_child_samples": [50, 100],
    "reg_lambda": [1.0, 5.0],
}

best_score = np.inf
best_params = None
best_iteration = None
best_alpha = None
best_model = None
best_val_predictions = None
trial_results = []

init_mlflow("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_Honest_Tuning"):
    for params in ParameterGrid(param_grid):
        model = lgb.LGBMRegressor(
            objective="regression_l1",
            n_estimators=2500,
            num_leaves=params["num_leaves"],
            learning_rate=params["learning_rate"],
            min_child_samples=params["min_child_samples"],
            reg_alpha=0.1,
            reg_lambda=params["reg_lambda"],
            max_depth=-1,
            max_bin=255,
            feature_fraction=0.85,
            bagging_fraction=0.85,
            bagging_freq=1,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=train_weights,
            categorical_feature=CAT_FEATURES,
            eval_set=[(X_val, y_val_residual)],
            eval_sample_weight=[val_weights],
            eval_metric="l1",
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        residual_prediction = model.predict(X_val, num_iteration=model.best_iteration_)

        # Choose how much of the learned correction to trust. alpha=0 is exactly
        # the seasonal baseline, so tuning cannot select a locally worse blend.
        alpha_scores = []
        for alpha in np.linspace(0.0, 1.20, 25):
            level_prediction = val_baseline + alpha * residual_prediction
            score = wmae(
                y_val_level.to_numpy(),
                level_prediction,
                validation_frame["IsHoliday"].to_numpy(),
            )
            alpha_scores.append((float(score), float(alpha)))

        score, alpha = min(alpha_scores, key=lambda x: x[0])
        trial_results.append({
            **params,
            "val_wmae": score,
            "alpha": alpha,
            "best_iteration": int(model.best_iteration_),
        })

        with mlflow.start_run(
            run_name=(
                f"leaves{params['num_leaves']}_lr{params['learning_rate']}"
                f"_child{params['min_child_samples']}_l2{params['reg_lambda']}"
            ),
            nested=True,
        ):
            mlflow.log_params(params)
            mlflow.log_param("correction_alpha", alpha)
            mlflow.log_metric("val_wmae", score)
            mlflow.log_metric("best_iteration", int(model.best_iteration_))

        if score < best_score:
            best_score = score
            best_params = params.copy()
            best_iteration = int(model.best_iteration_)
            best_alpha = alpha
            best_model = model
            best_val_predictions = val_baseline + alpha * residual_prediction

trial_results_df = pd.DataFrame(trial_results).sort_values("val_wmae").reset_index(drop=True)
print("Baseline WMAE:", round(baseline_wmae, 2))
print("Best model WMAE:", round(best_score, 2))
print("Best params:", best_params)
print("Best iteration:", best_iteration)
print("Best correction alpha:", best_alpha)
trial_results_df.head(10)

Accessing as akave23

Initialized MLflow to track repo "sansi23/Walmart-Recruiting---Store-Sales-Forecasting"

Repository sansi23/Walmart-Recruiting---Store-Sales-Forecasting initialized!

DagsHub connected: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow
Experiment: LightGBM_Training
Effective tracking URI: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow
🏃 View run leaves31_lr0.02_child50_l21.0 at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1/runs/9ee9be1d37a9423495860d33bf6fdaa6
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1
🏃 View run leaves31_lr0.02_child50_l25.0 at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1/runs/48621d4252584c6e813947c34f05df82
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1
🏃 View run leaves63_lr0.02_child50_l21.0 at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1/runs/82dcc5db189a4e519c57962cbb0

,learning_rate,min_child_samples,num_leaves,reg_lambda,val_wmae,alpha,best_iteration
0,0.02,50,63,1.0,1748.836104,0.75,96
1,0.02,100,63,1.0,1748.836104,0.75,96
2,0.02,100,31,1.0,1753.202258,0.80,94
3,0.02,50,31,1.0,1753.202258,0.80,94
4,0.02,100,63,5.0,1753.867891,0.80,77
5,0.04,50,63,1.0,1754.691616,0.75,40
6,0.04,100,63,1.0,1754.691616,0.75,40
7,0.02,50,63,5.0,1754.846007,0.85,74
8,0.04,100,31,5.0,1755.393297,0.80,43
9,0.04,50,31,5.0,1755.393297,0.80,43


## 8. Validation diagnostics

In [8]:
validation_diagnostic = validation_frame[[
    "Store", "Dept", "Date", "IsHoliday", TARGET, "Seasonal_Baseline"
]].copy()
validation_diagnostic["Prediction"] = best_val_predictions
validation_diagnostic["Absolute_Error"] = (
    validation_diagnostic[TARGET] - validation_diagnostic["Prediction"]
).abs()

weekly_scores = []
for date, group in validation_diagnostic.groupby("Date", sort=True):
    weekly_scores.append({
        "Date": date,
        "WMAE": wmae(
            group[TARGET].to_numpy(),
            group["Prediction"].to_numpy(),
            group["IsHoliday"].to_numpy(),
        ),
        "Rows": len(group),
    })
weekly_scores = pd.DataFrame(weekly_scores)

print("Worst validation weeks:")
display(weekly_scores.sort_values("WMAE", ascending=False).head(10))

importance = pd.Series(
    best_model.feature_importances_, index=LGBM_FEATURES
).sort_values(ascending=False)
display(importance.head(20).to_frame("importance"))

Worst validation weeks:


,Date,WMAE,Rows
9,2012-04-06,3083.991622,2983
11,2012-04-20,2845.698373,2975
12,2012-04-27,2254.131705,2954
30,2012-08-31,2031.213131,2962
15,2012-05-18,1936.938665,2953
29,2012-08-24,1925.439481,2960
22,2012-07-06,1893.931644,2961
8,2012-03-30,1843.240705,2961
18,2012-06-08,1825.636717,2960
1,2012-02-10,1771.052819,3001


,importance
Dept,1517
Store,1291
WeekOfYear,883
Seasonal_Baseline,639
Sales_Lag52_Origin,502
Sales_Roll_4w_Mean_Origin,326
Sales_Roll_52w_Mean_Origin,232
Sales_Roll_26w_Mean_Origin,173
Sales_Roll_13w_Mean_Origin,69
Size,51


## 9. Train the final model on all available historical origins

Hyperparameters and the number of boosting iterations come only from the honest validation experiment. The final model then uses every origin that has a complete 39-week target inside `train.csv`.

In [9]:
final_origin_train = build_origin_training_frame(
    train_merged,
    horizon_weeks=HORIZON,
    origin_stride_weeks=ORIGIN_STRIDE,
    min_history_weeks=52,
    require_full_horizon=True,
)

_exact_lag = train_merged[["Store", "Dept", "Date", TARGET]].copy()
_exact_lag["Date"] = _exact_lag["Date"] + pd.Timedelta(weeks=52)
_exact_lag = _exact_lag.rename(columns={TARGET: "_Exact_Lag52"})
final_origin_train = final_origin_train.merge(
    _exact_lag, on=["Store", "Dept", "Date"], how="left", validate="many_to_one"
)
final_origin_train["Sales_Lag52_Origin"] = final_origin_train["_Exact_Lag52"].combine_first(
    final_origin_train["Sales_Lag52_Origin"]
)
final_origin_train = final_origin_train.drop(columns="_Exact_Lag52")

final_origin_train["Week_Sin"] = np.sin(2 * np.pi * final_origin_train["WeekOfYear"] / 52.0)
final_origin_train["Week_Cos"] = np.cos(2 * np.pi * final_origin_train["WeekOfYear"] / 52.0)
final_origin_train["Month_Sin"] = np.sin(2 * np.pi * final_origin_train["Month"] / 12.0)
final_origin_train["Month_Cos"] = np.cos(2 * np.pi * final_origin_train["Month"] / 12.0)
final_origin_train["Horizon_Sin"] = np.sin(2 * np.pi * final_origin_train["Forecast_Horizon"] / HORIZON)
final_origin_train["Horizon_Cos"] = np.cos(2 * np.pi * final_origin_train["Forecast_Horizon"] / HORIZON)
final_origin_train["Lag52_Missing"] = final_origin_train["Sales_Lag52_Origin"].isna().astype(int)
final_origin_train["Seasonal_Baseline"] = (
    final_origin_train["Sales_Lag52_Origin"]
    .fillna(final_origin_train["Sales_Roll_52w_Mean_Origin"])
    .fillna(final_origin_train["Sales_Roll_26w_Mean_Origin"])
    .fillna(final_origin_train["Sales_Roll_13w_Mean_Origin"])
    .fillna(train_merged[TARGET].median())
)
final_origin_train["Residual_Target"] = (
    final_origin_train[TARGET] - final_origin_train["Seasonal_Baseline"]
)

X_final = final_origin_train[LGBM_FEATURES].copy()
for c in CAT_FEATURES:
    X_final[c] = X_final[c].astype("category")

y_final = final_origin_train["Residual_Target"].astype(float)
final_weights = np.where(final_origin_train["IsHoliday"].astype(bool), 5.0, 1.0)

# Robust correction limits stored for inference.
residual_clip_low = float(y_final.quantile(0.0025))
residual_clip_high = float(y_final.quantile(0.9975))

final_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=max(best_iteration, 50),
    num_leaves=best_params["num_leaves"],
    learning_rate=best_params["learning_rate"],
    min_child_samples=best_params["min_child_samples"],
    reg_alpha=0.1,
    reg_lambda=best_params["reg_lambda"],
    max_depth=-1,
    max_bin=255,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)
final_model.fit(
    X_final,
    y_final,
    sample_weight=final_weights,
    categorical_feature=CAT_FEATURES,
)

print("final rows:", len(final_origin_train))
print("final origins:", final_origin_train["Forecast_Origin_Date"].nunique())
print("residual clip:", residual_clip_low, residual_clip_high)

final rows: 3119718
final origins: 27
residual clip: -22265.01 22291.14


## 10. Generate Kaggle predictions with guaranteed row alignment

The `_input_row` column is carried through feature engineering and used to restore the original test order before constructing the submission. This is the critical fix for the 25K leaderboard score.

In [10]:
test_input = test_merged.copy()
assert test_input["_input_row"].is_unique

# Feature engineering is allowed to sort or merge internally because _input_row survives.
test_base = build_all_features(test_input, add_lags=False)
test_frame = add_origin_style_features(
    test_base,
    train_merged,
    origin_date=train_merged["Date"].max(),
)

_exact_lag = train_merged[["Store", "Dept", "Date", TARGET]].copy()
_exact_lag["Date"] = _exact_lag["Date"] + pd.Timedelta(weeks=52)
_exact_lag = _exact_lag.rename(columns={TARGET: "_Exact_Lag52"})
test_frame = test_frame.merge(
    _exact_lag, on=["Store", "Dept", "Date"], how="left", validate="many_to_one"
)
test_frame["Sales_Lag52_Origin"] = test_frame["_Exact_Lag52"].combine_first(
    test_frame["Sales_Lag52_Origin"]
)
test_frame = test_frame.drop(columns="_Exact_Lag52")

test_frame["Week_Sin"] = np.sin(2 * np.pi * test_frame["WeekOfYear"] / 52.0)
test_frame["Week_Cos"] = np.cos(2 * np.pi * test_frame["WeekOfYear"] / 52.0)
test_frame["Month_Sin"] = np.sin(2 * np.pi * test_frame["Month"] / 12.0)
test_frame["Month_Cos"] = np.cos(2 * np.pi * test_frame["Month"] / 12.0)
test_frame["Horizon_Sin"] = np.sin(2 * np.pi * test_frame["Forecast_Horizon"] / HORIZON)
test_frame["Horizon_Cos"] = np.cos(2 * np.pi * test_frame["Forecast_Horizon"] / HORIZON)
test_frame["Lag52_Missing"] = test_frame["Sales_Lag52_Origin"].isna().astype(int)
test_frame["Seasonal_Baseline"] = (
    test_frame["Sales_Lag52_Origin"]
    .fillna(test_frame["Sales_Roll_52w_Mean_Origin"])
    .fillna(test_frame["Sales_Roll_26w_Mean_Origin"])
    .fillna(test_frame["Sales_Roll_13w_Mean_Origin"])
    .fillna(train_merged[TARGET].median())
)

missing_test_features = [c for c in LGBM_FEATURES if c not in test_frame.columns]
assert not missing_test_features, missing_test_features
assert test_frame["_input_row"].is_unique
assert len(test_frame) == len(test_input)

X_test = test_frame[LGBM_FEATURES].copy()
for c in CAT_FEATURES:
    X_test[c] = X_test[c].astype("category")

test_correction = final_model.predict(X_test)
test_correction = np.clip(test_correction, residual_clip_low, residual_clip_high)
test_frame["Weekly_Sales_Pred"] = (
    test_frame["Seasonal_Baseline"].to_numpy() + best_alpha * test_correction
)

# CRITICAL: restore competition order after all sorting/merging.
test_frame = test_frame.sort_values("_input_row").reset_index(drop=True)
assert np.array_equal(test_frame["_input_row"].to_numpy(), np.arange(len(test_input)))

predictions = test_frame["Weekly_Sales_Pred"].to_numpy()
assert len(predictions) == len(test_raw)
assert np.isfinite(predictions).all()

submission = pd.DataFrame({
    "Id": (
        test_raw["Store"].astype(str) + "_" +
        test_raw["Dept"].astype(str) + "_" +
        pd.to_datetime(test_raw["Date"]).dt.strftime("%Y-%m-%d")
    ),
    "Weekly_Sales": predictions,
})
submission.to_csv("submission_lightgbm_fixed.csv", index=False)

print(submission.shape)
print(submission.head())
print(submission["Weekly_Sales"].describe())
print("saved: submission_lightgbm_fixed.csv")

(115064, 2)
               Id  Weekly_Sales
0  1_1_2012-11-02  40385.762056
1  1_1_2012-11-09  19324.875339
2  1_1_2012-11-16  19579.122696
3  1_1_2012-11-23  21481.931942
4  1_1_2012-11-30  25575.728425
count    115064.000000
mean      16691.136626
std       23975.948618
min       -1800.406912
25%        2129.429136
50%        7736.391994
75%       21090.742539
max      648712.360226
Name: Weekly_Sales, dtype: float64
saved: submission_lightgbm_fixed.csv


## 11. Strong row-alignment audit

In [11]:
# Join predictions back to test keys by the preserved row position and verify exact identity.
alignment_audit = test_frame[["_input_row", "Store", "Dept", "Date", "Weekly_Sales_Pred"]].merge(
    test_input[["_input_row", "Store", "Dept", "Date"]],
    on="_input_row",
    how="inner",
    suffixes=("_pred", "_input"),
    validate="one_to_one",
)

assert (alignment_audit["Store_pred"] == alignment_audit["Store_input"]).all()
assert (alignment_audit["Dept_pred"] == alignment_audit["Dept_input"]).all()
assert (alignment_audit["Date_pred"] == alignment_audit["Date_input"]).all()
print("Row-alignment audit passed for all", len(alignment_audit), "test rows.")

Row-alignment audit passed for all 115064 test rows.


## 12. MLflow logging and deployable pipeline

In [12]:
with mlflow.start_run(run_name="LightGBM_Final_Fixed") as final_run:
    mlflow.log_params({
        **best_params,
        "n_estimators": max(best_iteration, 50),
        "correction_alpha": best_alpha,
        "horizon": HORIZON,
        "origin_stride": ORIGIN_STRIDE,
        "target": "Weekly_Sales_minus_seasonal_baseline",
        "row_order_fix": True,
    })
    mlflow.log_metric("validation_wmae", float(best_score))
    mlflow.log_metric("seasonal_baseline_wmae", float(baseline_wmae))
    mlflow.log_metric("improvement_pct", float(100 * (baseline_wmae - best_score) / baseline_wmae))

    importance = pd.Series(final_model.feature_importances_, index=LGBM_FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(8, 9))
    importance.tail(25).plot.barh(ax=ax)
    ax.set_title("Fixed LightGBM feature importance")
    fig.tight_layout()
    fig.savefig("lightgbm_fixed_importance.png", dpi=140)
    mlflow.log_artifact("lightgbm_fixed_importance.png")
    mlflow.log_artifact("submission_lightgbm_fixed.csv")
    final_run_id = final_run.info.run_id

joblib.dump(final_model, "lgbm_fixed_model.joblib")
train_merged[["Store", "Dept", "Date", TARGET]].to_parquet("lgbm_history.parquet", index=False)

pipeline_config = {
    "feature_cols": LGBM_FEATURES,
    "cat_features": CAT_FEATURES,
    "horizon": HORIZON,
    "correction_alpha": float(best_alpha),
    "residual_clip_low": residual_clip_low,
    "residual_clip_high": residual_clip_high,
    "global_median": float(train_merged[TARGET].median()),
}
with open("lgbm_fixed_config.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_config, f, indent=2)

print("Final MLflow run:", final_run_id)

🏃 View run LightGBM_Final_Fixed at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1/runs/d48f7e959fd1458cbd58fdd3ef10005c
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1
Final MLflow run: d48f7e959fd1458cbd58fdd3ef10005c


In [13]:
class WalmartLGBMFixedPipeline(mlflow.pyfunc.PythonModel):
    """Merged Walmart test rows in; correctly aligned Weekly_Sales predictions out."""

    def load_context(self, context):
        self.model = joblib.load(context.artifacts["model"])
        self.history = pd.read_parquet(context.artifacts["history"])
        with open(context.artifacts["config"], "r", encoding="utf-8") as f:
            self.config = json.load(f)

    def predict(self, context, model_input: pd.DataFrame):
        df = model_input.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df["_input_row"] = np.arange(len(df), dtype=np.int64)

        base = build_all_features(df, add_lags=False)
        feat = add_origin_style_features(
            base,
            self.history,
            origin_date=self.history["Date"].max(),
        )

        exact_lag = self.history[["Store", "Dept", "Date", TARGET]].copy()
        exact_lag["Date"] = exact_lag["Date"] + pd.Timedelta(weeks=52)
        exact_lag = exact_lag.rename(columns={TARGET: "_Exact_Lag52"})
        feat = feat.merge(
            exact_lag, on=["Store", "Dept", "Date"], how="left", validate="many_to_one"
        )
        feat["Sales_Lag52_Origin"] = feat["_Exact_Lag52"].combine_first(
            feat["Sales_Lag52_Origin"]
        )
        feat = feat.drop(columns="_Exact_Lag52")

        horizon = int(self.config["horizon"])
        feat["Week_Sin"] = np.sin(2 * np.pi * feat["WeekOfYear"] / 52.0)
        feat["Week_Cos"] = np.cos(2 * np.pi * feat["WeekOfYear"] / 52.0)
        feat["Month_Sin"] = np.sin(2 * np.pi * feat["Month"] / 12.0)
        feat["Month_Cos"] = np.cos(2 * np.pi * feat["Month"] / 12.0)
        feat["Horizon_Sin"] = np.sin(2 * np.pi * feat["Forecast_Horizon"] / horizon)
        feat["Horizon_Cos"] = np.cos(2 * np.pi * feat["Forecast_Horizon"] / horizon)
        feat["Lag52_Missing"] = feat["Sales_Lag52_Origin"].isna().astype(int)
        feat["Seasonal_Baseline"] = (
            feat["Sales_Lag52_Origin"]
            .fillna(feat["Sales_Roll_52w_Mean_Origin"])
            .fillna(feat["Sales_Roll_26w_Mean_Origin"])
            .fillna(feat["Sales_Roll_13w_Mean_Origin"])
            .fillna(float(self.config["global_median"]))
        )

        X = feat[self.config["feature_cols"]].copy()
        for c in self.config["cat_features"]:
            X[c] = X[c].astype("category")

        correction = self.model.predict(X)
        correction = np.clip(
            correction,
            float(self.config["residual_clip_low"]),
            float(self.config["residual_clip_high"]),
        )
        feat["prediction"] = (
            feat["Seasonal_Baseline"].to_numpy()
            + float(self.config["correction_alpha"]) * correction
        )

        # The essential fix: predictions are restored to input position.
        feat = feat.sort_values("_input_row")
        assert np.array_equal(feat["_input_row"].to_numpy(), np.arange(len(df)))
        return pd.Series(
            feat["prediction"].to_numpy(),
            index=model_input.index,
            name="Weekly_Sales_Pred",
        )

with mlflow.start_run(run_name="LightGBM_Fixed_ModelRegistration") as reg_run:
    mlflow.log_param("source_final_run_id", final_run_id)
    mlflow.pyfunc.log_model(
        name="walmart_lgbm_fixed_pipeline",
        python_model=WalmartLGBMFixedPipeline(),
        artifacts={
            "model": "lgbm_fixed_model.joblib",
            "history": "lgbm_history.parquet",
            "config": "lgbm_fixed_config.json",
        },
        registered_model_name="WalmartLightGBMFixedPipeline",
    )
    registration_run_id = reg_run.info.run_id

print("Registration run:", registration_run_id)

2026/07/11 23:32:10 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/11 23:32:10 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


Successfully registered model 'WalmartLightGBMFixedPipeline'.
2026/07/11 23:32:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: WalmartLightGBMFixedPipeline, version 1
Created version '1' of model 'WalmartLightGBMFixedPipeline'.


🏃 View run LightGBM_Fixed_ModelRegistration at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1/runs/d6d60aeafb5d45b4bcdbc447e9de4430
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/1
Registration run: d6d60aeafb5d45b4bcdbc447e9de4430


## 13. Reloaded-pipeline equivalence test

In [15]:
from mlflow import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(
    "name='WalmartLightGBMFixedPipeline'"
)

latest_version = max(int(v.version) for v in versions)

loaded_pipeline = mlflow.pyfunc.load_model(
    f"models:/WalmartLightGBMFixedPipeline/{latest_version}"
)

# Use the full test frame because some feature engineering operations
# depend on having the complete forecast horizon available.
full_test_input = (
    test_merged
    .drop(columns=["_input_row"], errors="ignore")
    .copy()
)

loaded_predictions = loaded_pipeline.predict(full_test_input)

assert len(loaded_predictions) == len(full_test_input)
assert loaded_predictions.index.equals(full_test_input.index)
assert np.isfinite(loaded_predictions.to_numpy()).all()

np.testing.assert_allclose(
    loaded_predictions.to_numpy(),
    predictions,
    rtol=1e-7,
    atol=1e-5,
)

print(
    f"Loaded pipeline version {latest_version} exactly reproduced "
    f"all {len(full_test_input):,} direct test predictions."
)

Loaded pipeline version 1 exactly reproduced all 115,064 direct test predictions.


## Conclusion

The approximately 25K Kaggle score was not normal model generalization error. The primary failure was **submission-row misalignment after preprocessing reordered the test frame**. The fixed notebook preserves a positional key through every transformation and restores original test order before returning predictions.

The revised validation is also more trustworthy because it reproduces the real task: one forecast origin followed by 39 completely unseen weeks. The lag-52 residual formulation and holiday-weighted fitting should normally improve stability beyond merely fixing the row order.